In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split



In [2]:
df = pd.read_csv('../data/pre_processed_data_week3.csv')
X_df = df.drop('fraud_reported', axis=1)
y = df['fraud_reported'].values.reshape(-1, 1)

X_df = X_df.apply(pd.to_numeric, errors='coerce').fillna(0)


model_columns = X_df.columns.tolist()

import pickle
with open("../data/model_columns_week4.pkl", "wb") as f:
    pickle.dump(model_columns, f)

X = X_df.values
X = (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-8)




In [3]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
n_features = X_train.shape[1]
weights = np.zeros((n_features, 1))
bias = 0
learning_rate = 0.0005
n_iterations = 8000



In [4]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [5]:
def compute_class_weights_adjusted(y, balance_factor=0.9):
    total = len(y)
    pos = np.sum(y == 1)
    neg = np.sum(y == 0)
    weight_for_0 = total / (2 * neg) * balance_factor
    weight_for_1 = total / (2 * pos) * (2 - balance_factor)
    return weight_for_0, weight_for_1

In [6]:
weight_0, weight_1 = compute_class_weights_adjusted(y_train, balance_factor=0.9)

for i in range(n_iterations):
    z = np.dot(X_train, weights) + bias
    y_pred = sigmoid(z)
    dz = y_pred - y_train
    dz[y_train == 0] *= weight_0
    dz[y_train == 1] *= weight_1
    dw = np.dot(X_train.T, dz) / len(y_train)
    db = np.sum(dz) / len(y_train)
    weights -= learning_rate * dw
    bias -= learning_rate * db

In [7]:
def predict(X, weights, bias, threshold=0.5):
    y_pred_prob = sigmoid(np.dot(X, weights) + bias)
    return (y_pred_prob >= threshold).astype(int), y_pred_prob

y_pred_train, train_probs = predict(X_train, weights, bias)
train_accuracy = np.mean(y_pred_train == y_train)

y_pred_test, test_probs = predict(X_test, weights, bias)


In [8]:
thresholds = np.linspace(0.45, 0.55, 21)
best_f1 = 0
best_thresh = 0.5
for t in thresholds:
    y_temp = (test_probs >= t).astype(int)
    TP = np.sum((y_test==1) & (y_temp==1))
    FP = np.sum((y_test==0) & (y_temp==1))
    FN = np.sum((y_test==1) & (y_temp==0))
    precision = TP / (TP + FP + 1e-8)
    recall = TP / (TP + FN + 1e-8)
    f1 = 2*(precision*recall)/(precision+recall+1e-8)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t

In [9]:
y_pred_test_best = (test_probs >= best_thresh).astype(int)
TP = np.sum((y_test==1) & (y_pred_test_best==1))
TN = np.sum((y_test==0) & (y_pred_test_best==0))
FP = np.sum((y_test==0) & (y_pred_test_best==1))
FN = np.sum((y_test==1) & (y_pred_test_best==0))

In [10]:
accuracy = (TP + TN)/(TP + TN + FP + FN)
precision = TP / (TP + FP + 1e-8)
recall = TP / (TP + FN + 1e-8)
f1_score = 2*(precision*recall)/(precision+recall+1e-8)

print(f"Best Threshold: {best_thresh:.2f}")
print(f"Training Accuracy: {train_accuracy*100:.2f}%")
print(f"Test Accuracy: {accuracy*100:.2f}%")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-score: {f1_score:.2f}")


Best Threshold: 0.49
Training Accuracy: 40.71%
Test Accuracy: 39.19%
Precision: 0.29
Recall: 0.99
F1-score: 0.44


In [10]:

np.savez('../data/logistic_model_week4.npz', weights=weights, bias=bias)
pd.DataFrame(X_train, columns=X_df.columns).to_csv('../data/X_train_week4.csv', index=False)
pd.DataFrame(y_train, columns=['fraud_reported']).to_csv('../data/y_train_week4.csv', index=False)


In [11]:
pd.DataFrame(X_test, columns=X_df.columns).to_csv('../data/X_test_week4.csv', index=False)
pd.DataFrame(y_test, columns=['fraud_reported']).to_csv('../data/y_test_week4.csv', index=False)
